# Recorded pitcher-pickoff response

Frozen protocol comparison on chronology-v2 data. 2024 is development; 2025 remains closed. Predictions are associations under observed strategies, not causal effects or recommendations to throw over.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT))
from scripts.artifact_runs import resolve_run
from scripts.pitcher_response import validate
run=resolve_run(ROOT/'results/pitcher_response',ROOT/'data/processed/pitcher_response')
validate(run)
metrics=pd.read_csv(run.results/'metrics.csv')
for filename,keys in [('calibration.csv',['split','variant','model']),('subgroups.csv',['split','variant','model','subgroup'])]:
    for key,g in pd.read_csv(run.results/filename).groupby(keys):
        row=metrics[(metrics.split==key[0])&(metrics.variant==key[1])&(metrics.model==key[2])].iloc[0]
        assert g.n.sum()==row.n and g.events.sum()==row.attempts
assert not (ROOT/'data/historical_feed_manifest_2025.csv').exists()
pd.read_csv(run.results/'sample_flow.csv')

Out[1]: 
              split            variant   fit_n  ...  runner_change  unknown  catcher
0   diagnostic_2023            primary   69255  ...           1262        4      164
1   diagnostic_2023  calibrated_threat   69255  ...           1262        4      164
2   diagnostic_2023      history_depth   69255  ...           1262        4      164
3   diagnostic_2023    known_initiator   69252  ...           1262        0      164
4  development_2024            primary  138862  ...           2532        9      381
5  development_2024  calibrated_threat  138862  ...           2532        9      381
6  development_2024      history_depth  138862  ...           2532        9      381
7  development_2024    known_initiator  138855  ...           2532        0      380

[8 rows x 12 columns]


,split,variant,fit_n,fit_games,fit_events,n,games,events,terminal,runner_change,unknown,catcher
0,diagnostic_2023,primary,69255,1217,3845,68864,1201,3598,1881,1262,4,164
1,diagnostic_2023,calibrated_threat,69255,1217,3845,68864,1201,3598,1881,1262,4,164
2,diagnostic_2023,history_depth,69255,1217,3845,68864,1201,3598,1881,1262,4,164
3,diagnostic_2023,known_initiator,69252,1217,3843,68860,1201,3594,1877,1262,0,164
4,development_2024,primary,138862,2430,7479,136487,2429,7633,3911,2532,9,381
5,development_2024,calibrated_threat,138862,2430,7479,136487,2429,7633,3911,2532,9,381
6,development_2024,history_depth,138862,2430,7479,136487,2429,7633,3911,2532,9,381
7,development_2024,known_initiator,138855,2430,7473,136478,2429,7625,3902,2532,0,380


In [2]:
metrics[metrics.variant.eq('primary')]

Out[2]: 
               split  variant  ...     brier  average_precision
0    diagnostic_2023  primary  ...  0.049529           0.052248
1    diagnostic_2023  primary  ...  0.049218           0.084093
2    diagnostic_2023  primary  ...  0.048288           0.121026
12  development_2024  primary  ...  0.052801           0.055925
13  development_2024  primary  ...  0.052669           0.078172
14  development_2024  primary  ...  0.051807           0.114492

[6 rows x 12 columns]


,split,variant,model,n,attempts,mean_prediction,observed_rate,calibration_intercept,calibration_slope,log_loss,brier,average_precision
0,diagnostic_2023,primary,constant,68864,3598,0.055519,0.052248,NaN,NaN,0.205186,0.049529,0.052248
1,diagnostic_2023,primary,context,68864,3598,0.055285,0.052248,-1.055255,0.633912,0.202108,0.049218,0.084093
2,diagnostic_2023,primary,context_threat,68864,3598,0.055321,0.052248,-0.565115,0.804489,0.192757,0.048288,0.121026
12,development_2024,primary,constant,136487,7633,0.053859,0.055925,NaN,NaN,0.215645,0.052801,0.055925
13,development_2024,primary,context,136487,7633,0.053466,0.055925,-1.180476,0.559280,0.214467,0.052669,0.078172
14,development_2024,primary,context_threat,136487,7633,0.053243,0.055925,-0.612648,0.747655,0.205546,0.051807,0.114492


In [3]:
pd.read_csv(run.results/'bootstrap.csv').query("variant == 'primary'")

Out[3]: 
               split  variant     cluster  ...     delta     lower     upper
0    diagnostic_2023  primary     game_pk  ... -0.009351 -0.010544 -0.008247
1    diagnostic_2023  primary     game_pk  ... -0.000929 -0.001178 -0.000665
2    diagnostic_2023  primary     game_pk  ...  0.036934  0.030799  0.043085
3    diagnostic_2023  primary  pitcher_id  ... -0.009351 -0.010542 -0.008053
4    diagnostic_2023  primary  pitcher_id  ... -0.000929 -0.001200 -0.000646
5    diagnostic_2023  primary  pitcher_id  ...  0.036934  0.030884  0.043484
6    diagnostic_2023  primary   runner_id  ... -0.009351 -0.011035 -0.007811
7    diagnostic_2023  primary   runner_id  ... -0.000929 -0.001210 -0.000647
8    diagnostic_2023  primary   runner_id  ...  0.036934  0.029811  0.045825
36  development_2024  primary     game_pk  ... -0.008921 -0.009774 -0.008056
37  development_2024  primary     game_pk  ... -0.000862 -0.001050 -0.000684
38  development_2024  primary     game_pk  ...  0.036320  0.033158 

,split,variant,cluster,metric,delta,lower,upper
0,diagnostic_2023,primary,game_pk,log_loss,-0.009351,-0.010544,-0.008247
1,diagnostic_2023,primary,game_pk,brier,-0.000929,-0.001178,-0.000665
2,diagnostic_2023,primary,game_pk,average_precision,0.036934,0.030799,0.043085
3,diagnostic_2023,primary,pitcher_id,log_loss,-0.009351,-0.010542,-0.008053
4,diagnostic_2023,primary,pitcher_id,brier,-0.000929,-0.001200,-0.000646
5,diagnostic_2023,primary,pitcher_id,average_precision,0.036934,0.030884,0.043484
6,diagnostic_2023,primary,runner_id,log_loss,-0.009351,-0.011035,-0.007811
7,diagnostic_2023,primary,runner_id,brier,-0.000929,-0.001210,-0.000647
8,diagnostic_2023,primary,runner_id,average_precision,0.036934,0.029811,0.045825
36,development_2024,primary,game_pk,log_loss,-0.008921,-0.009774,-0.008056


In [4]:
metrics[metrics.split.eq('development_2024')]

Out[4]: 
               split            variant  ...     brier  average_precision
12  development_2024            primary  ...  0.052801           0.055925
13  development_2024            primary  ...  0.052669           0.078172
14  development_2024            primary  ...  0.051807           0.114492
15  development_2024  calibrated_threat  ...  0.052801           0.055925
16  development_2024  calibrated_threat  ...  0.052669           0.078172
17  development_2024  calibrated_threat  ...  0.051804           0.113903
18  development_2024      history_depth  ...  0.052801           0.055925
19  development_2024      history_depth  ...  0.052579           0.082404
20  development_2024      history_depth  ...  0.051846           0.115418
21  development_2024    known_initiator  ...  0.052753           0.055870
22  development_2024    known_initiator  ...  0.052637           0.077972
23  development_2024    known_initiator  ...  0.051757           0.114887

[12 rows x 12 columns]


,split,variant,model,n,attempts,mean_prediction,observed_rate,calibration_intercept,calibration_slope,log_loss,brier,average_precision
12,development_2024,primary,constant,136487,7633,0.053859,0.055925,NaN,NaN,0.215645,0.052801,0.055925
13,development_2024,primary,context,136487,7633,0.053466,0.055925,-1.180476,0.559280,0.214467,0.052669,0.078172
14,development_2024,primary,context_threat,136487,7633,0.053243,0.055925,-0.612648,0.747655,0.205546,0.051807,0.114492
15,development_2024,calibrated_threat,constant,136487,7633,0.053859,0.055925,NaN,NaN,0.215645,0.052801,0.055925
16,development_2024,calibrated_threat,context,136487,7633,0.053466,0.055925,-1.180476,0.559280,0.214467,0.052669,0.078172
17,development_2024,calibrated_threat,context_threat,136487,7633,0.052333,0.055925,-0.613570,0.741968,0.205789,0.051804,0.113903
18,development_2024,history_depth,constant,136487,7633,0.053859,0.055925,NaN,NaN,0.215645,0.052801,0.055925
19,development_2024,history_depth,context,136487,7633,0.055827,0.055925,-1.084530,0.601564,0.213430,0.052579,0.082404
20,development_2024,history_depth,context_threat,136487,7633,0.055442,0.055925,-0.668178,0.738920,0.205350,0.051846,0.115418
21,development_2024,known_initiator,constant,136478,7625,0.053819,0.055870,NaN,NaN,0.215489,0.052753,0.055870
